## 테스트용

In [3]:
!pip install neo4j-graphrag neo4j openai mysql-connector-python pandas gradio


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import json
from getpass import getpass

import pandas as pd
import mysql.connector
import gradio as gr

from neo4j import GraphDatabase, basic_auth
from neo4j.time import Date

from neo4j_graphrag.retrievers import Text2CypherRetriever
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.generation import GraphRAG
import re
import time

import requests
from math import radians, sin, cos, sqrt, atan2

c:\Users\SSAFY\Documents\Wedding\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "20010910"

In [6]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=basic_auth(NEO4J_USER, NEO4J_PASSWORD)
)

In [7]:
with driver.session() as session:
    count_result = session.run("MATCH (n) RETURN count(n) AS cnt")
    print("Neo4j node count:", count_result.single()["cnt"])

Neo4j node count: 29273


In [8]:
import re

def parse_section_title(text):

    result = {
        "benefit_rate_min": None,
        "benefit_rate_max": None,
        "benefit_amount": None,
        "benefit_unit": None,
        "benefit_type": None
    }

    if not text:
        return result

    # % 범위
    m = re.search(r'(\d+(\.\d+)?)~(\d+(\.\d+)?)\s*%', text)
    if m:
        result["benefit_rate_min"] = float(m.group(1))
        result["benefit_rate_max"] = float(m.group(3))
        result["benefit_unit"] = "percent"

    # 단일 %
    m = re.search(r'(\d+(\.\d+)?)\s*%', text)
    if m and result["benefit_rate_min"] is None:
        result["benefit_rate_min"] = float(m.group(1))
        result["benefit_rate_max"] = float(m.group(1))
        result["benefit_unit"] = "percent"

    # 원
    m = re.search(r'([\d,]+)\s*원', text)
    if m:
        result["benefit_amount"] = int(m.group(1).replace(",", ""))
        result["benefit_unit"] = "krw"

    # 리터당
    m = re.search(r'리터\s*당\s*([\d,]+)\s*원', text)
    if m:
        result["benefit_amount"] = int(m.group(1).replace(",", ""))
        result["benefit_unit"] = "krw_per_liter"

    # 혜택 유형
    if "할인" in text:
        result["benefit_type"] = "discount"
    elif "적립" in text:
        result["benefit_type"] = "point"
    elif "캐시백" in text:
        result["benefit_type"] = "cashback"

    return result

In [9]:
parse_section_title("백화점/할인점/홈쇼핑·온라인몰 1.0~5.0% 적립")

{'benefit_rate_min': 1.0,
 'benefit_rate_max': 5.0,
 'benefit_amount': None,
 'benefit_unit': 'percent',
 'benefit_type': 'discount'}

In [10]:
MERCHANT_CATEGORY_MAP = {
    "스타벅스": "cafe",
    "투썸": "cafe",
    "이디야": "cafe",
    "CU": "convenience_store",
    "GS25": "convenience_store",
    "CGV": "movie",
    "이마트": "mart"
}

In [11]:
def get_candidate_cards(tx, category):

    query = """
    MATCH (c:Card)-[:HAS_BENEFIT]->(b:Benefit)-[:IN_CATEGORY]->(cat:Category)
    WHERE cat.name = $category
    RETURN c.card_id AS card_id,
           c.name AS card_name,
           b.benefit_type AS benefit_type,
           b.benefit_value AS benefit_value,
           b.benefit_unit AS benefit_unit
    """

    result = tx.run(query, category=category)

    return [dict(r) for r in result]

In [12]:
def calculate_benefit(amount, benefit):

    unit = benefit["benefit_unit"]
    value = benefit["benefit_value"]

    if unit == "percent":
        return amount * (value / 100)

    if unit == "krw":
        return value

    return 0

In [13]:
def recommend_card(driver, merchant, amount):

    category = MERCHANT_CATEGORY_MAP.get(merchant)

    if not category:
        return None

    with driver.session() as session:
        cards = session.execute_read(get_candidate_cards, category)

    best_card = None
    best_value = 0

    for card in cards:

        benefit = calculate_benefit(amount, card)

        if benefit > best_value:
            best_value = benefit
            best_card = card

    return {
        "recommended_card": best_card,
        "expected_benefit": best_value
    }

In [22]:
recommend_card(driver, "스타벅스", 300000)

{'recommended_card': {'card_id': 660,
  'card_name': 'LIKIT FUN+ 카드',
  'benefit_type': 'discount',
  'benefit_value': 60.0,
  'benefit_unit': 'percent'},
 'expected_benefit': 180000.0}